In [ ]:
# View raw nfl data
import pandas as pd
df = pd.read_parquet("../data/raw/schedules.parquet")
pd.set_option("display.max_columns", None)
df.head()        # first 5 rows
df.columns       # all column names
df.shape         # (rows, columns)
df[df["season"] == 2025].head(20)   # filter to a season

model comparisons

In [2]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.elo import EloModel
from nflmodel.models.ridge import RidgeModel
from nflmodel.models.lgbm import LGBMModel
from nflmodel.models.massey import MasseyModel
from nflmodel.config import BACKTEST_SEASONS

models = {
    "elo": EloModel(),
    "massey": MasseyModel(),
    "ridge": RidgeModel(),
    "lgbm": LGBMModel(),
}

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    for name, model in models.items():
        test[name] = model.fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

# how similar are the predictions? (1.0 = identical)
print(preds[["ridge", "lgbm", "elo", "massey"]].corr().round(3))

# how similar are the MISTAKES? This is what matters for combining models
errors = preds[["ridge", "lgbm", "elo", "massey"]].sub(preds["result"], axis=0)
print(errors.corr().round(3))

        ridge   lgbm    elo  massey
ridge   1.000  0.966  0.944   0.951
lgbm    0.966  1.000  0.947   0.950
elo     0.944  0.947  1.000   0.939
massey  0.951  0.950  0.939   1.000
        ridge   lgbm    elo  massey
ridge   1.000  0.993  0.989   0.990
lgbm    0.993  1.000  0.991   0.990
elo     0.989  0.991  1.000   0.988
massey  0.990  0.990  0.988   1.000


ridge weight list

In [4]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.ridge import RidgeModel

df = build_features()
model = RidgeModel().fit(df)              # all completed games, same as the live model

ridge = model.pipe[-1]                    # the Ridge step inside the pipeline
weights = pd.Series(ridge.coef_, index=model.features)
print(weights.sort_values(key=abs, ascending=False).round(2))
print("intercept:", round(ridge.intercept_, 2))

massey_margin         2.50
elo_margin            1.77
home_off_epa_prior    0.93
away_qb_starts       -0.87
away_off_epa_prior   -0.83
home_qb_starts        0.82
home_def_epa_prior   -0.69
rest_diff             0.38
away_def_epa_prior   -0.23
home_field            0.21
dtype: float64
intercept: 2.13


In [5]:
import numpy as np
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.hybrid import HybridModel
from nflmodel.config import BACKTEST_SEASONS

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    test["pred"] = HybridModel().fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

edge = preds["pred"] - preds["spread_line"]       # + means we like the home team more than Vegas
cover = preds["result"] - preds["spread_line"]    # + means the home team covered

for threshold in [0, 2, 3, 4, 5]:
    bets = preds[(edge.abs() >= threshold) & (edge != 0)]
    e, c = edge[bets.index], cover[bets.index]
    won = np.sign(e) == np.sign(c)
    push = c == 0
    profit = np.where(push, 0, np.where(won, 0.909, -1.0))
    decided = (~push).sum()
    print(f"edge >= {threshold}: {len(bets):4d} bets, "
          f"win rate {won[~push].mean():.1%}, "
          f"profit {profit.sum():+.1f} units, ROI {profit.sum() / len(bets):+.1%}")

edge >= 0: 3028 bets, win rate 51.9%, profit -24.6 units, ROI -0.8%
edge >= 2: 1357 bets, win rate 52.3%, profit -2.1 units, ROI -0.2%
edge >= 3:  817 bets, win rate 51.7%, profit -10.6 units, ROI -1.3%
edge >= 4:  419 bets, win rate 51.7%, profit -5.3 units, ROI -1.3%
edge >= 5:  219 bets, win rate 49.8%, profit -10.7 units, ROI -4.9%


In [6]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.ridge import RidgeModel
from nflmodel.models.lgbm import LGBMModel
from nflmodel.config import BACKTEST_SEASONS

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    test["ridge"] = RidgeModel().fit(train).predict(test)
    test["lgbm"] = LGBMModel().fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

for w in [1.0, 0.8, 0.7, 0.5]:
    blend = w * preds["ridge"] + (1 - w) * preds["lgbm"]
    print(f"ridge weight {w}: MAE {(blend - preds['result']).abs().mean():.3f}")

ridge weight 1.0: MAE 10.067
ridge weight 0.8: MAE 10.065
ridge weight 0.7: MAE 10.065
ridge weight 0.5: MAE 10.071


Testing

In [ ]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.ridge import RidgeModel

df = build_features()
model = RidgeModel().fit(df[df["season"] < 2024])

ridge = model.pipe[-1]          # the Ridge step inside the pipeline
weights = pd.Series(ridge.coef_, index=model.features)
print(weights.sort_values(key=abs, ascending=False).round(2))
print("intercept:", round(ridge.intercept_, 2))

In [ ]:
import nflreadpy as nfl

inj = nfl.load_injuries([2026]).to_pandas()
depth = nfl.load_depth_charts([2026]).to_pandas()

# 1. How fresh is each? (latest week / latest snapshot)
print(inj[inj["season"] == 2026]["week"].max())
print(depth["dt"].max())

In [ ]:
# 2. Current QB1 for every team
qb1 = depth[(depth["pos_abb"] == "QB") & (depth["pos_rank"] == 1)]
latest = qb1[qb1["dt"] == qb1["dt"].max()]
print(latest[["team", "player_name", "gsis_id"]].sort_values("team").to_string(index=False))

In [ ]:
# 3. Do the IDs and team codes match the schedule?
from nflmodel.data.games import load_games
g = load_games()
last = g[g["season"] == 2026].tail(16)
print(last[["home_team", "home_qb_name", "home_qb_id"]].head())

In [ ]:
tb = g[(g["home_team"] == "TB") | (g["away_team"] == "TB")].tail(3)
print(tb[["week", "home_team", "home_qb_name", "away_team", "away_qb_name"]].to_string(index=False))

In [ ]:
from nflmodel.features.build import build_features
import pandas as pd
df = build_features()

rows = pd.concat([
    df[["season", "week", "home_team", "home_qb_id"]].set_axis(["season", "week", "team", "qb"], axis=1),
    df[["season", "week", "away_team", "away_qb_id"]].set_axis(["season", "week", "team", "qb"], axis=1),
]).sort_values(["team", "season", "week"])
rows["changed"] = rows.groupby("team")["qb"].shift(1) != rows["qb"]
print(rows["changed"].mean())
print(rows[rows["week"] > 1]["changed"].mean())

In [ ]:
df = build_features(include_upcoming=True)
up = df[df["result"].isna() & (df["week"] == 6) & (df["season"] == 2026)]
print(up[["game_id", "home_off_epa_prior", "away_off_epa_prior",
          "home_qb_id", "away_qb_id", "home_qb_starts", "away_qb_starts", "elo_margin"]].to_string(index=False))

In [ ]:
import nflreadpy as nfl

depth = nfl.load_depth_charts([2026]).to_pandas()
darnold = depth[depth["player_name"].str.contains("Darnold", na=False)]
print(darnold[["team", "player_name", "gsis_id", "pos_rank"]].drop_duplicates())